In [5]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 1. Load the data
df = pd.read_csv('../data/synthetic_health_data.csv')

# 2. Data Transformation: Interest -> Transactions
# We assume interest >= 7 counts as a "Likely Purchase" for the IA
basket = pd.DataFrame()
basket['purchased_Ambrotose'] = (df['interest_Ambrotose'] >= 7).astype(int)
basket['purchased_OSP'] = (df['interest_OSP'] >= 7).astype(int)

# 3. Add variety: Simulate interest in other common products 
# (This ensures we have enough "itemsets" to find patterns)
import numpy as np
np.random.seed(42)
basket['purchased_Omega3'] = np.random.choice([0, 1], size=len(df), p=[0.7, 0.3])
basket['purchased_Probiotics'] = np.random.choice([0, 1], size=len(df), p=[0.6, 0.4])

print("Check purchase counts (must be > 0):")
print(basket.sum())

# 4. Find Frequent Itemsets using FP-Growth
# Threshold at 0.01 (10 out of 1000 people)
basket = basket.astype(bool)

frequent_itemsets = fpgrowth(basket, min_support=0.01, use_colnames=True)

if frequent_itemsets.empty:
    print("Error: Still no patterns found. Check your data logic.")
else:
    # 5. Generate Association Rules
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
    
    # Clean and sort for the IA's report
    rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    rules = rules.sort_values('lift', ascending=False)

    print(f"\n--- Discovered {len(rules)} Association Rules ---")
    print(rules.head(10))

Check purchase counts (must be > 0):
purchased_Ambrotose     370
purchased_OSP           213
purchased_Omega3        288
purchased_Probiotics    430
dtype: int64

--- Discovered 28 Association Rules ---
                                          antecedents  \
25                                    (purchased_OSP)   
18  (purchased_Ambrotose, purchased_Probiotics, pu...   
10            (purchased_Ambrotose, purchased_Omega3)   
13                                    (purchased_OSP)   
23                  (purchased_OSP, purchased_Omega3)   
20        (purchased_Ambrotose, purchased_Probiotics)   
22              (purchased_OSP, purchased_Probiotics)   
21            (purchased_Ambrotose, purchased_Omega3)   
9                                     (purchased_OSP)   
8         (purchased_Ambrotose, purchased_Probiotics)   

                                          consequents  support  confidence  \
25  (purchased_Ambrotose, purchased_Probiotics, pu...    0.010    0.046948   
18           